In [312]:
# Importamos librerias
import yfinance as yf
import numpy as np
import pandas as pd
import math
from scipy.optimize import minimize

In [313]:
# Lista de acciones y fecha de descarga de los datos
acciones = ['AAPL', 'JPM', 'XOM', 'JNJ', 'PG', 'CAT', 'NEE', 'AMT', 'DIS', 'HD']
fecha_inicio = '2020-1-1'
fecha_final = '2026-7-28'
# Retorno esperado anualizado para minimizar la varianza (en decimales)
re_minimo = .20
# Riesgo esperado para maximizar los retornos (en decimales)
riesgo_maximo= .20

In [314]:
# Obtener tasa libre de riesgo de los bonos del tesoro de USA a 5y a la fecha dada al inicio
rf = yf.download(tickers='^FVX', start=fecha_inicio, end=fecha_final)['Close']
rf = float(rf.iloc[-1])/100
rf

[*********************100%***********************]  1 of 1 completed
C:\Users\jcbro\AppData\Local\Temp\ipykernel_43532\3187470673.py:3: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  rf = float(rf.iloc[-1])/100


0.04396999835968018

In [315]:
# Usar api de yfinance para descargar el precio de los tickers y extraer solo la columna 'Close'
df_acciones = (yf.download(tickers=acciones, start= fecha_inicio, end= fecha_final, multi_level_index= False))['Close']
df_acciones

[*********************100%***********************]  10 of 10 completed


Ticker,AAPL,AMT,CAT,DIS,HD,JNJ,JPM,NEE,PG,XOM
Date,,,,,,,,,,
2020-01-02,72.333862,189.909729,132.067871,143.689270,187.322601,121.936958,117.899185,50.714516,103.959305,52.947037
2020-01-03,71.630638,190.001144,130.234207,142.041000,186.700073,120.525200,116.343361,51.075813,103.260109,52.521358
2020-01-06,72.201401,189.951279,130.146454,141.216858,187.578430,120.374870,116.250885,51.330853,103.403313,52.924637
2020-01-07,71.861862,185.903748,128.426895,141.265366,186.350433,121.109978,114.274506,51.286224,102.763100,52.491501
2020-01-08,73.017830,187.516129,129.567413,140.974472,189.139038,121.093246,115.165985,51.262840,103.201134,51.699902
...,...,...,...,...,...,...,...,...,...,...
2026-07-21,327.739990,163.389999,889.969971,96.139999,331.600006,250.610001,345.230011,87.930000,147.002625,151.710007
2026-07-22,325.890015,166.059998,889.309998,95.870003,331.450012,255.630005,348.209991,89.410004,148.024994,154.449997
2026-07-23,321.660004,164.649994,894.539978,92.830002,324.709991,259.269989,349.899994,89.790001,145.880997,156.889999


In [316]:
# Crear la lista de acciones ahora como columnas para evitar errores de acomodo
acciones = np.array(df_acciones.columns)
acciones


array(['AAPL', 'AMT', 'CAT', 'DIS', 'HD', 'JNJ', 'JPM', 'NEE', 'PG',
       'XOM'], dtype=object)

In [317]:
# Calcular rendimiento diario de las acciones
df_rendimiento = ((df_acciones/df_acciones.shift(1))-1).dropna()
df_rendimiento

Ticker,AAPL,AMT,CAT,DIS,HD,JNJ,JPM,NEE,PG,XOM
Date,,,,,,,,,,
2020-01-03,-0.009722,0.000481,-0.013884,-0.011471,-0.003323,-0.011578,-0.013196,0.007124,-0.006726,-0.008040
2020-01-06,0.007968,-0.000262,-0.000674,-0.005802,0.004705,-0.001247,-0.000795,0.004993,0.001387,0.007678
2020-01-07,-0.004703,-0.021308,-0.013212,0.000343,-0.006547,0.006107,-0.017001,-0.000869,-0.006191,-0.008184
2020-01-08,0.016086,0.008673,0.008881,-0.002059,0.014964,-0.000138,0.007801,-0.000456,0.004263,-0.015081
2020-01-09,0.021241,0.003767,-0.002505,-0.003920,0.015330,0.002967,0.003651,0.007836,0.010938,0.007656
...,...,...,...,...,...,...,...,...,...,...
2026-07-21,0.003521,-0.021968,0.029700,-0.002801,-0.004324,0.007194,0.018768,-0.000795,-0.006907,0.022580
2026-07-22,-0.005645,0.016341,-0.000742,-0.002808,-0.000452,0.020031,0.008632,0.016832,0.006955,0.018061
2026-07-23,-0.012980,-0.008491,0.005881,-0.031710,-0.020335,0.014239,0.004853,0.004250,-0.014484,0.015798


In [318]:
# Retorno anualizado
re_anualizado = df_rendimiento.mean()*252
re_anualizado

Ticker
AAPL    0.284486
AMT     0.023882
CAT     0.344733
DIS    -0.006718
HD      0.128822
JNJ     0.138688
JPM     0.216420
NEE     0.129257
PG      0.076137
XOM     0.216993
dtype: float64

In [319]:
# Matriz de covarianza anualizada
cov_anualizada= df_rendimiento.cov()*252
cov_anualizada

Ticker,AAPL,AMT,CAT,DIS,HD,JNJ,JPM,NEE,PG,XOM
Ticker,,,,,,,,,,
AAPL,0.098566,0.033879,0.035610,0.045148,0.045086,0.018701,0.039587,0.033142,0.024879,0.026729
AMT,0.033879,0.087678,0.021235,0.026216,0.038768,0.027505,0.029270,0.047446,0.029813,0.019710
CAT,0.035610,0.021235,0.111583,0.048700,0.038518,0.017087,0.060720,0.027106,0.014243,0.052729
DIS,0.045148,0.026216,0.048700,0.108155,0.043389,0.015900,0.056199,0.029339,0.019541,0.042470
HD,0.045086,0.038768,0.038518,0.043389,0.077990,0.020635,0.040784,0.037271,0.026798,0.024729
JNJ,0.018701,0.027505,0.017087,0.015900,0.020635,0.038935,0.020593,0.024849,0.023522,0.017344
JPM,0.039587,0.029270,0.060720,0.056199,0.040784,0.020593,0.094875,0.028636,0.020232,0.050909
NEE,0.033142,0.047446,0.027106,0.029339,0.037271,0.024849,0.028636,0.086829,0.030296,0.025505
PG,0.024879,0.029813,0.014243,0.019541,0.026798,0.023522,0.020232,0.030296,0.042966,0.013249


In [320]:
# Funcion retorno del portafolio
def retorno_portafolio(pesos):
    return sum(re_anualizado*pesos)
# Funcion riesgo del portafolio
def riesgo_portafolio(pesos):
    return (pesos @ cov_anualizada @ pesos)**(1/2)


In [321]:
# Confirmar que ambas funciones funcionan correctamente
# Creamos contador de acciones
n = 0
for t in acciones:
    n = n+1
# Tranformamos a array los pesos para evitar errores
pesos_prueba = np.array([1 / n] * n)
# Imprimimos los resultados de las funciones al pasar el argumento de pesos_prueba
print(f'N0 acciones: {n} \npesos prueba: {pesos_prueba}')
print(f'retorno esperado del portafolio: {retorno_portafolio(pesos_prueba)}')
print(f'riesgo del portafolio: {riesgo_portafolio(pesos_prueba)}')

N0 acciones: 10 
pesos prueba: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
retorno esperado del portafolio: 0.15527006239770427
riesgo del portafolio: 0.19186976100696532


In [322]:
# Crear funcion que calcula el sharpe ratio dado los pesos
def sharpe_ratio(pesos):
    return (retorno_portafolio(pesos)-rf)/riesgo_portafolio(pesos)

In [323]:
# Verificar si la funcion de sharpe ratio funciona correctamente
print(sharpe_ratio(pesos_prueba))

0.5800813189837852


In [324]:
# Creamos una restriccion de igualdad en la cual la suma de los pesos -1 tiene que ser igual a 0 (nos aseguramos que los pesos sumen 1)
# Esto se hace para evitar que el optimizador realice ventas en corto
restriccion_pesos = {
    'type': 'eq',
    'fun': lambda pesos: sum(pesos)-1
}
# Creamos un bound en el cual el peso de las acciones tienen que estar entre 0 y 1 (`n` lo creamos anteriormente y contiene el numero de acciones)
limite_pesos = [(0, 1) for x in range(n)]    
limite_pesos

[(0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1),
 (0, 1)]

In [325]:
# Creamos punto de partida por medio de un array distribuyendo los pesos uniformemente y que su suma sea igual a 1
x0 = np.array([1/len(acciones)]*len(acciones))
x0

array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

In [326]:
# Funcion de sharpe ratio negativo ya que el equivalente a maximizar el sharpe ratio usando minimize (que solo minimiza) es pasandolo como negativo
def sharpe_ratio_negativo(pesos):
    return -sharpe_ratio(pesos)

### Portafolio maximo sharpe ratio

In [327]:
# Dentro de la variable resultado se almacenan los pesos optimos por accion para maximizar el sharpe ratio
portafolio_sharpe = minimize(fun= sharpe_ratio_negativo, x0=x0, method='SLSQP', bounds=limite_pesos, constraints=restriccion_pesos)

In [328]:
# Observamos los resultados del optimizador
portafolio_sharpe

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -1.0353441304310462
       x: [ 3.452e-01  5.624e-17  4.602e-01  0.000e+00  0.000e+00
            1.702e-01  0.000e+00  0.000e+00  0.000e+00  2.434e-02]
     nit: 7
     jac: [-1.883e-01  4.033e-01 -1.883e-01  8.212e-01  1.585e-01
           -1.882e-01 -4.670e-02 -7.371e-03  4.360e-02 -1.884e-01]
    nfev: 77
    njev: 7

In [329]:
# Resumen de resultados en DataFrame
df_port_sharpe = pd.DataFrame(portafolio_sharpe.x)
df_port_sharpe.index = acciones 
df_port_sharpe.columns = ['Pesos optimos por accion %']
df_port_sharpe = round(df_port_sharpe,4)*100
df_port_sharpe

,Pesos optimos por accion %
AAPL,34.52
AMT,0.00
CAT,46.02
DIS,0.00
HD,0.00
JNJ,17.02
JPM,0.00
NEE,0.00
PG,0.00
XOM,2.43


In [330]:
# Observar Sharpe Ratio, retorno esperado y riesgo del portafolio optimizado
print(f'sharpe ratio = {sharpe_ratio(portafolio_sharpe.x)}')
print(f'retorno esperado = {retorno_portafolio(portafolio_sharpe.x)}')
print(f'riesgo portafolio = {riesgo_portafolio(portafolio_sharpe.x)}')

sharpe ratio = 1.0353441304310462
retorno esperado = 0.2857575625299336
riesgo portafolio = 0.23353352480936912


### Portafolio de mínimo riesgo dado un retorno objetivo

In [331]:
# Creamos restriccion de not equal (>=0) en la cual expresamos que el portafolio por lo menos tiene que dar el retorno minimo esperado
restriccion_retorno_esperado = {
    'type': 'ineq',
    'fun': lambda pesos: sum(pesos*re_anualizado)-re_minimo
}

In [332]:
# Ahora en fun en vez de la funcion de sharpe ratio, pasamos la funcion de riesgo_portafolio y agregamos la restriccion nueva
portafolio_min_var = minimize(fun=riesgo_portafolio , x0=x0, method='SLSQP', bounds=limite_pesos, 
                              constraints=[restriccion_pesos, restriccion_retorno_esperado])
portafolio_min_var

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 0.17731420263303962
       x: [ 1.672e-01  0.000e+00  1.867e-01  0.000e+00  1.802e-17
            4.300e-01  9.758e-19  0.000e+00  1.311e-01  8.502e-02]
     nit: 10
     jac: [ 2.066e-01  1.525e-01  2.285e-01  1.671e-01  1.647e-01
            1.559e-01  1.905e-01  1.546e-01  1.336e-01  1.830e-01]
    nfev: 111
    njev: 10

In [333]:
# Resumen de resultados en DataFrame
df_port_var = pd.DataFrame(portafolio_min_var.x)
df_port_var.index = acciones 
df_port_var.columns = ['Pesos optimos por accion %']
df_port_var = round(df_port_var,4)*100
df_port_var

,Pesos optimos por accion %
AAPL,16.72
AMT,0.00
CAT,18.67
DIS,0.00
HD,0.00
JNJ,43.00
JPM,0.00
NEE,0.00
PG,13.11
XOM,8.50


In [334]:
# Observar Sharpe Ratio, retorno esperado y riesgo del portafolio optimizado
print(f'sharpe ratio = {sharpe_ratio(portafolio_min_var.x)}')
print(f'retorno esperado = {retorno_portafolio(portafolio_min_var.x)}')
print(f'riesgo portafolio = {riesgo_portafolio(portafolio_min_var.x)}')

sharpe ratio = 0.8799633606445308
retorno esperado = 0.19999999999865506
riesgo portafolio = 0.17731420263303962


### Portafolio de maximo retorno dado un nivel de riesgo maximo

In [335]:
# Funcion de rendimiento negativo (misma logica que con el sharpe ratio)
def retorno_negativo(pesos):
    return -retorno_portafolio(pesos)

In [336]:
# Creamos restriccion de not equal (>=0) en la cual expresamos que el portafolio por lo menos tiene que dar el retorno minimo esperado
restriccion_max_riesgo = {
    'type': 'ineq',
    'fun': lambda pesos: riesgo_maximo - (pesos @ cov_anualizada @ pesos)**(1/2)
}

In [337]:
# Ahora en fun pasamaos la funcion de retorno anualizado del portafolio
portafolio_max_re = minimize(fun=retorno_negativo, x0=x0, method='SLSQP', bounds=limite_pesos, 
                              constraints=[restriccion_pesos, restriccion_max_riesgo])
portafolio_max_re 

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.24546161696129948
       x: [ 2.615e-01  0.000e+00  3.104e-01  0.000e+00  0.000e+00
            3.683e-01  8.007e-17  0.000e+00  0.000e+00  5.974e-02]
     nit: 7
     jac: [-2.845e-01 -2.388e-02 -3.447e-01  6.718e-03 -1.288e-01
           -1.387e-01 -2.164e-01 -1.293e-01 -7.614e-02 -2.170e-01]
    nfev: 77
    njev: 7

In [338]:
# Resumen de resultados en DataFrame
df_port_ret = pd.DataFrame(portafolio_max_re.x)
df_port_ret.index = acciones 
df_port_ret.columns = ['Pesos optimos por accion %']
df_port_ret = round(df_port_ret,4)*100
df_port_var

,Pesos optimos por accion %
AAPL,16.72
AMT,0.00
CAT,18.67
DIS,0.00
HD,0.00
JNJ,43.00
JPM,0.00
NEE,0.00
PG,13.11
XOM,8.50


In [339]:
# Observar Sharpe Ratio, retorno esperado y riesgo del portafolio optimizado
print(f'sharpe ratio = {sharpe_ratio(portafolio_max_re.x)}')
print(f'retorno esperado = {retorno_portafolio(portafolio_max_re.x)}')
print(f'riesgo portafolio = {riesgo_portafolio(portafolio_max_re.x)}')

sharpe ratio = 1.0074555028233523
retorno esperado = 0.24546161696129948
riesgo portafolio = 0.20000051420330467
